# Imeplementación de modelos de Redes Neuronales Profundas

En este archivo, voy a poner en práctica lo que hemos ido aprendiendo a lo largo de la asignatura tanto en la parte de toería como en la parte de práctica. La idea es crear una CNN la cual sea capaz de analizar fotos de paciente con Pneumonia y sin está; esto nos va a permitir entrenar el modelo ver cómo actúa y si es capaz de funcionar de manera correcta. 

**Realizado por Rodrigo Gálvez Travalja**

## Importación de las librerias 

In [1]:
%pip install torch torchvision 
%pip install kagglehub
%pip install matplotlib
%pip install pandas

import kagglehub    
import pandas as pd
import heapq
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
%pip install torchvision 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image
from PIL import Image


import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("cuda runtime version:", torch.version.cuda)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/Users/rodrigo/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Note: you may need to restart the kernel to use updated packages.
torch version: 2.11.0
cuda available: False


# Comprobación de GPU y configuración de `device`
# Esta celda verifica si PyTorch detecta CUDA y crea la variable `device`.


In [2]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("cuda runtime version:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)


def move_to_device(batch, device):
    """Mueve tensores dentro de `batch` al `device`.
    Soporta `torch.Tensor`, tuplas/listas y diccionarios.
    """
    if isinstance(batch, (list, tuple)):
        return type(batch)(move_to_device(x, device) for x in batch)
    if isinstance(batch, dict):
        return {k: move_to_device(v, device) for k, v in batch.items()}
    try:
        return batch.to(device)
    except Exception:
        return batch


torch version: 2.11.0
cuda available: False
using device: cpu


## Descarga de los archivos de Kaggle 

In [3]:
ruta = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
archivos = os.listdir(ruta)
# Mostrar el contenido de la ruta descargada
for archivo in archivos:
    print(archivo)


ruta_100x100 = os.path.join(ruta, 'chest_xray', 'train', 'NORMAL')
os.chdir(ruta_100x100)
print(f"Directorio actual: {os.getcwd()}")

# Listar el contenido del directorio correctamente
for carpeta in os.listdir(ruta_100x100):
    ruta_carpeta = os.path.join(ruta_100x100, carpeta)
    print(f"Carpeta: {carpeta}")
    if os.path.isdir(ruta_carpeta):  # Verificar si es un directorio
        print(os.listdir(ruta_carpeta))
# --- Descargar dataset desde KaggleHub ---
directorio = os.path.join(ruta, 'chest_xray')
RUTA_ENTRENAMIENTO = os.path.join(directorio, "train")
RUTA_PRUEBA = os.path.join(directorio, "Test")  
        

chest_xray
Directorio actual: /Users/rodrigo/.cache/kagglehub/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray/train/NORMAL
Carpeta: NORMAL2-IM-0927-0001.jpeg
Carpeta: NORMAL2-IM-1056-0001.jpeg
Carpeta: IM-0427-0001.jpeg
Carpeta: NORMAL2-IM-1260-0001.jpeg
Carpeta: IM-0656-0001-0001.jpeg
Carpeta: IM-0561-0001.jpeg
Carpeta: NORMAL2-IM-1110-0001.jpeg
Carpeta: IM-0757-0001.jpeg
Carpeta: NORMAL2-IM-1326-0001.jpeg
Carpeta: NORMAL2-IM-0736-0001.jpeg
Carpeta: NORMAL2-IM-0500-0001.jpeg
Carpeta: NORMAL2-IM-0393-0001.jpeg
Carpeta: NORMAL2-IM-0994-0001.jpeg
Carpeta: IM-0207-0001.jpeg
Carpeta: IM-0494-0001.jpeg
Carpeta: IM-0177-0001.jpeg
Carpeta: IM-0388-0001.jpeg
Carpeta: IM-0341-0001.jpeg
Carpeta: IM-0355-0001.jpeg
Carpeta: IM-0449-0001.jpeg
Carpeta: IM-0480-0001.jpeg
Carpeta: NORMAL2-IM-1038-0001.jpeg
Carpeta: NORMAL2-IM-1348-0001.jpeg
Carpeta: IM-0739-0001.jpeg
Carpeta: IM-0213-0001.jpeg
Carpeta: NORMAL2-IM-0452-0001.jpeg
Carpeta: NORMAL2-IM-0980-0001.jpeg
Carpeta: NORMAL2-

## Preprocesamiento de imagenes 

In [ ]:
TAMANIO_IMAGEN = (100, 100) #Tamaño de la imagen (ancho, alto)

transformacion_entrenamiento = transforms.Compose([ #Transformaciones para el conjunto de entrenamiento
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), #Ajuste de brillo, contraste, saturación y matiz
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], #Normalización de la imagen con la media y desviación estándar de los canales RGB
                         std=[0.229, 0.224, 0.225])
])

transformacion_val_test = transforms.Compose([ #Transformaciones para el conjunto de validación y prueba
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class CarpetaImagenesPersonalizada(Dataset): #Clase personalizada para cargar las imágenes desde una carpeta
    def __init__(self, directorio_raiz, transformacion=None):
        self.conjunto_datos = datasets.ImageFolder(directorio_raiz, transform=transformacion)

    def __len__(self):
        return len(self.conjunto_datos)

    def __getitem__(self, idx):
        imagen, etiqueta = self.conjunto_datos[idx]
        ruta_archivo = self.conjunto_datos.imgs[idx][0]
        return imagen, etiqueta, ruta_archivo

def cargar_datos(ruta_entrenamiento, ruta_prueba): #Función para cargar los datos de entrenamiento y prueba
    torch.manual_seed(42)
    conjunto_entrenamiento = CarpetaImagenesPersonalizada(ruta_entrenamiento, transformacion=transformacion_entrenamiento)
    conjunto_prueba = CarpetaImagenesPersonalizada(ruta_prueba, transformacion=transformacion_val_test)
    tam_entrenamiento = int(0.8 * len(conjunto_entrenamiento))
    tam_validacion = len(conjunto_entrenamiento) - tam_entrenamiento
    datos_entrenamiento, datos_validacion = torch.utils.data.random_split(conjunto_entrenamiento, [tam_entrenamiento, tam_validacion])
    cargador_entrenamiento = DataLoader(datos_entrenamiento, batch_size=32, shuffle=True)
    cargador_validacion = DataLoader(datos_validacion, batch_size=32, shuffle=False)
    cargador_prueba = DataLoader(conjunto_prueba, batch_size=32, shuffle=False)
    return cargador_entrenamiento, cargador_validacion, cargador_prueba


## Creacion de la red neuronal convolucional

In [ ]:
# Paso 2: Red neuronal convolucional
class Modelo_CNN(nn.Module):
    def __init__(self, num_classes):
        super(Modelo_CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 12 * 12, 128)  # Ajustar el tamaño según la salida real después del pooling
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)  # Aplanar la imagen dinámicamente
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## Entrenamiento y valiacion de la red neuronal

In [ ]:
# Paso 3: Entrenamiento y evaluación
def entrenar_modelo(modelo, cargador_entrenamiento, cargador_validacion, epocas=5, learning_rate=0.001):
    # Definir el optimizador y la función de pérdida
    optimizador = optim.RMSprop(modelo.parameters(), lr=learning_rate)
    criterio = nn.CrossEntropyLoss()

    # Guardar las pérdidas y precisiones para visualización
    perdidas_entrenamiento, perdidas_validacion = [], []
    precisiones_entrenamiento, precisiones_validacion = [], []

    for epoca in range(epocas):
        modelo.train()
        perdida_acumulada = 0.0
        correctos_entrenamiento = 0
        total_entrenamiento = 0
        
        for entradas, etiquetas, _ in cargador_entrenamiento:
            optimizador.zero_grad()
            salidas = modelo(entradas.to(torch.float32))  # Asegurarse de que las entradas sean del tipo correcto
            perdida = criterio(salidas, etiquetas)
            perdida.backward()
            optimizador.step()

            perdida_acumulada += perdida.item()
            _, predicho = torch.max(salidas.data, 1)
            total_entrenamiento += etiquetas.size(0)
            correctos_entrenamiento += (predicho == etiquetas).sum().item()

        perdidas_entrenamiento.append(perdida_acumulada / len(cargador_entrenamiento))
        precisiones_entrenamiento.append(100 * correctos_entrenamiento / total_entrenamiento)

        # Evaluación en conjunto de validación
        modelo.eval()
        perdida_val_acumulada = 0.0
        correctos_val = 0
        total_val = 0
        
        with torch.no_grad():
            for entradas, etiquetas, _ in cargador_validacion:
                salidas = modelo(entradas)
                perdida = criterio(salidas, etiquetas)
                perdida_val_acumulada += perdida.item()
                _, predicho = torch.max(salidas.data, 1)
                total_val += etiquetas.size(0)
                correctos_val += (predicho == etiquetas).sum().item()

        perdidas_validacion.append(perdida_val_acumulada / len(cargador_validacion))
        precisiones_validacion.append(100 * correctos_val / total_val)

        print(f"Epoch {epoca+1}/{epocas}, Loss: {perdida_acumulada/len(cargador_entrenamiento):.4f}, Accuracy: {100 * correctos_entrenamiento / total_entrenamiento:.2f}%")
        print(f"Validation Loss: {perdida_val_acumulada/len(cargador_validacion):.4f}, Validation Accuracy: {100 * correctos_val / total_val:.2f}%")
        
        if perdidas_entrenamiento[-1] <= 0.2: #Condición para detener el entrenamiento, como se nos pide en el enunciado 
            print(f"Entrenamiento detenido en época {epoca+1} porque la pérdida de entrenamiento es ≤ 0.2.")
            break
    return modelo, perdidas_entrenamiento, perdidas_validacion, precisiones_entrenamiento, precisiones_validacion



## Prediccion y visualizacion de los resultados 

In [ ]:
# Paso 4: Predicción y visualización de resultados
def predecir_imagenes(modelo, cargador_prueba, etiquetas):
    modelo.eval()
    predicciones = {}
    imagenes_mostradas = 0
    max_imagenes = 10

    with torch.no_grad():
        for entradas,etiquetas, _ in cargador_prueba:
            salidas = modelo(entradas)
            _, predicho = torch.max(salidas.data, 1)

            for idx in range(len(predicho)):
                clase = etiquetas[predicho[idx]]
                if clase == "PNEUMONIA" and imagenes_mostradas < max_imagenes:
                    imagen = to_pil_image(entradas[idx].cpu().detach())
                    plt.imshow(imagen)
                    plt.title(f"Predicción: {clase}")
                    plt.axis("off")
                    plt.show()
                    imagenes_mostradas += 1

                if clase not in predicciones:
                    predicciones[clase] = 1
                else:
                    predicciones[clase] += 1

    return predicciones

# Cargar datos de entrenamiento y prueba
directorio = os.path.join(ruta, "chest_xray")
os.chdir(directorio)
RUTA_ENTRENAMIENTO = os.path.join(directorio, "train")
RUTA_PRUEBA = os.path.join(directorio, "Test")
cargador_entrenamiento, cargador_validacion, cargador_prueba = cargar_datos(RUTA_ENTRENAMIENTO, RUTA_PRUEBA)
#Obtenemos las clases directamente desde el dataset
conjunto_datos = datasets.ImageFolder(RUTA_ENTRENAMIENTO, transform=transformacion_entrenamiento)
clases_conjunto = conjunto_datos.classes  #Listado de clases en el orden correcto
numero_clases = len(clases_conjunto)
modelo = Modelo_CNN(numero_clases)
modelo, perdidas_entrenamiento, perdidas_validacion, precisiones_entrenamiento, precisiones_validacion = entrenar_modelo(
    modelo, cargador_entrenamiento, cargador_validacion
)

# Obtener las predicciones y mostrar los resultados
predicciones = predecir_imagenes(modelo, cargador_prueba, sorted(os.listdir(RUTA_ENTRENAMIENTO)))

# Mostrar la categoría predicha para cada carpeta de imágenes
print("\nCategorías predichas para cada carpeta de imágenes:")
for category, count in predicciones.items():
    print(f"{category}: {count} imágenes clasificadas como {category} ")

def mostrar_graficas_entrenamiento(perdidas_entrenamiento, perdidas_validacion, precisiones_entrenamiento, precisiones_validacion): #Función para mostrar gráficas de pérdida y precisión
    epocas = range(1, len(perdidas_entrenamiento) + 1)
    #Gráfica de pérdidas
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epocas, perdidas_entrenamiento, label='Entrenamiento')
    plt.plot(epocas, perdidas_validacion, label='Validación')
    plt.title('Pérdida durante el entrenamiento')
    plt.xlabel('Épocas')
    plt.ylabel('Pérdida')
    plt.legend()
    #Gráfica de precisiones
    plt.subplot(1, 2, 2)
    plt.plot(epocas, precisiones_entrenamiento, label='Entrenamiento')
    plt.plot(epocas, precisiones_validacion, label='Validación')
    plt.title('Precisión durante el entrenamiento')
    plt.xlabel('Épocas')
    plt.ylabel('Precisión (%)')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
#Mostramos gráficas de pérdida y precisión
mostrar_graficas_entrenamiento(perdidas_entrenamiento, perdidas_validacion, precisiones_entrenamiento, precisiones_validacion)

## Prediccion del modelo

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from PIL import Image
import glob
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

# --------------------- CONFIGURACIÓN ---------------------
ruta_datos = os.path.join(ruta, 'chest_xray')
TAMANIO_IMAGEN = (100, 100)

transformacion = transforms.Compose([
    transforms.Resize(TAMANIO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

nombres_etiquetas = ["NORMAL", "PNEUMONIA_bacterial", "PNEUMONIA_viral"]

# --------------------- DATASET PERSONALIZADO ---------------------
class DatasetPersonalizadoRayosX(Dataset):
    def __init__(self, directorio_raiz, transformacion=None):
        self.rutas_imagenes = []
        self.etiquetas = []
        self.transformacion = transformacion
        self.mapa_etiquetas = {
            'NORMAL': 0,
            'PNEUMONIA_bacterial': 1,
            'PNEUMONIA_viral': 2
        }

        imagenes_normales = glob.glob(os.path.join(directorio_raiz, "NORMAL", "*.jpeg"))
        imagenes_neumonia = glob.glob(os.path.join(directorio_raiz, "PNEUMONIA", "*.jpeg"))

        for ruta_imagen in imagenes_normales:
            self.rutas_imagenes.append(ruta_imagen)
            self.etiquetas.append(self.mapa_etiquetas['NORMAL'])

        for ruta_imagen in imagenes_neumonia:
            nombre_fichero = os.path.basename(ruta_imagen).lower()
            if "bacteria" in nombre_fichero:
                self.rutas_imagenes.append(ruta_imagen)
                self.etiquetas.append(self.mapa_etiquetas['PNEUMONIA_bacterial'])
            elif "virus" in nombre_fichero:
                self.rutas_imagenes.append(ruta_imagen)
                self.etiquetas.append(self.mapa_etiquetas['PNEUMONIA_viral'])

    def __len__(self):
        return len(self.rutas_imagenes)

    def __getitem__(self, idx):
        imagen = Image.open(self.rutas_imagenes[idx]).convert("RGB")
        etiqueta = self.etiquetas[idx]

        if self.transformacion:
            imagen = self.transformacion(imagen)

        return imagen, etiqueta

# --------------------- CARGA Y DIVISIÓN DE DATOS ---------------------
dataset_entrenamiento_completo = DatasetPersonalizadoRayosX(os.path.join(ruta_datos, "train"), transformacion=transformacion)
indices_entrenamiento, indices_validacion = train_test_split(
    list(range(len(dataset_entrenamiento_completo))),
    test_size=0.2,
    stratify=dataset_entrenamiento_completo.etiquetas,
    random_state=42
)

dataset_entrenamiento = data.Subset(dataset_entrenamiento_completo, indices_entrenamiento)
dataset_validacion = data.Subset(dataset_entrenamiento_completo, indices_validacion)
dataset_prueba = DatasetPersonalizadoRayosX(os.path.join(ruta_datos, "test"), transformacion=transformacion)

loader_entrenamiento = DataLoader(dataset_entrenamiento, batch_size=32, shuffle=True)
loader_validacion = DataLoader(dataset_validacion, batch_size=32, shuffle=False)
loader_prueba = DataLoader(dataset_prueba, batch_size=32, shuffle=False)

# --------------------- MODELO ---------------------
class ModeloCNN(nn.Module):
    def __init__(self, num_clases):
        super(ModeloCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, num_clases)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# --------------------- ENTRENAMIENTO ---------------------
def entrenar_modelo(modelo, loader_entrenamiento, loader_validacion, num_epocas=5, tasa_aprendizaje=0.001):
    optimizador = optim.Adam(modelo.parameters(), lr=tasa_aprendizaje)
    criterio = nn.CrossEntropyLoss()

    perdidas_entrenamiento, perdidas_validacion = [], []
    precisiones_entrenamiento, precisiones_validacion = [], []

    for epoca in range(num_epocas):
        modelo.train()
        perdida_corriente = 0.0
        correctos_entrenamiento = 0
        total_entrenamiento = 0

        for entradas, etiquetas in loader_entrenamiento:
            optimizador.zero_grad()
            salidas = modelo(entradas)
            perdida = criterio(salidas, etiquetas)
            perdida.backward()
            optimizador.step()

            perdida_corriente += perdida.item()
            _, predicho = torch.max(salidas.data, 1)
            total_entrenamiento += etiquetas.size(0)
            correctos_entrenamiento += (predicho == etiquetas).sum().item()

        perdidas_entrenamiento.append(perdida_corriente / len(loader_entrenamiento))
        precisiones_entrenamiento.append(100 * correctos_entrenamiento / total_entrenamiento)

        # Validación
        modelo.eval()
        perdida_corriente_validacion = 0.0
        correctos_validacion = 0
        total_validacion = 0

        with torch.no_grad():
            for entradas, etiquetas in loader_validacion:
                salidas = modelo(entradas)
                perdida = criterio(salidas, etiquetas)
                perdida_corriente_validacion += perdida.item()
                _, predicho = torch.max(salidas.data, 1)
                total_validacion += etiquetas.size(0)
                correctos_validacion += (predicho == etiquetas).sum().item()

        perdidas_validacion.append(perdida_corriente_validacion / len(loader_validacion))
        precisiones_validacion.append(100 * correctos_validacion / total_validacion)

        print(f"Época {epoca+1}/{num_epocas}, Pérdida: {perdidas_entrenamiento[-1]:.4f}, Precisión: {precisiones_entrenamiento[-1]:.2f}%")
        print(f"Pérdida Validación: {perdidas_validacion[-1]:.4f}, Precisión Validación: {precisiones_validacion[-1]:.2f}%")

    return modelo, perdidas_entrenamiento, perdidas_validacion, precisiones_entrenamiento, precisiones_validacion

# --------------------- PREDICCIÓN ---------------------
def predecir_imagenes_con_probabilidades(modelo, loader_prueba, nombres_etiquetas, max_imagenes=5):
    modelo.eval()
    imagenes_mostradas = 0

    with torch.no_grad():
        for entradas, _ in loader_prueba:
            salidas = modelo(entradas)
            probabilidades = F.softmax(salidas, dim=1)

            for i in range(entradas.size(0)):
                if imagenes_mostradas >= max_imagenes:
                    return

                probs = probabilidades[i].cpu().numpy()
                imagen = to_pil_image(entradas[i].cpu())
                clase_predicha = nombres_etiquetas[np.argmax(probs)]

                plt.imshow(imagen)
                plt.axis("off")
                plt.title(f"Predicción: {clase_predicha}")
                plt.show()

                top_indices = np.argsort(probs)[::-1]
                print("Probabilidades por clase:")
                for idx in top_indices:
                    print(f"{nombres_etiquetas[idx]}: {probs[idx]*100:.2f}%")
                print("-" * 40)
                imagenes_mostradas += 1

# --------------------- EJECUCIÓN ---------------------
num_clases = 3
modelo = ModeloCNN(num_clases)

modelo, perdidas_entrenamiento, perdidas_validacion, precisiones_entrenamiento, precisiones_validacion = entrenar_modelo(
    modelo, loader_entrenamiento, loader_validacion, num_epocas=5, tasa_aprendizaje=0.001
)

predecir_imagenes_con_probabilidades(modelo, loader_prueba, nombres_etiquetas, max_imagenes=5)
